# LCA Lasers4MaaS 



In [79]:
import bw2data as bd
import bw2io as bi
from bw2calc import LCA
import pandas as pd
import ipywidgets as widgets
import numpy as np
from IPython.display import display

In [80]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"

bd.projects.set_current(PROJECT_NAME)
if ECOINVENT_DATABASE_NAME not in bd.databases:
    bi.import_ecoinvent_release(version="3.12", system_model="cutoff")
else:
    print(f"Database {ECOINVENT_DATABASE_NAME} already exists, skipping import.")

Database ecoinvent-3.12-cutoff already exists, skipping import.


In [81]:
ei = bd.Database(ECOINVENT_DATABASE_NAME)

In [82]:
bio = bd.Database("ecoinvent-3.12-biosphere")

### Global parameters

In [83]:
# Technosphere exchanges
elec = [a for a in ei.search("market group for electricity, medium voltage") if a.get('location') == "RER"][0]
# wire = [a for a in ei.search("welding wire") if a.get('location') == "GLO"] # a proxy needs to be found
gas  = [a for a in ei.search("market for argon") if a.get('location') == "RER"][0]

# Biosphere exchanges
pm25 = bio.search("Particulate matter, < 2.5 um")[0]

In [84]:
# Cost factors for the main resource inputs (adjust as needed)
COST_FACTORS = {
    'electricity_EUR_per_kWh': 0.29,     # EUR 0.29/kWh for industrial electricity
    'argon_EUR_per_kg': 5.87,           # EUR 2.5/kg for argon gas
    'nitrogen_EUR_per_kg': 5.35           # EUR 1.2/kg for nitrogen gas
}

In [85]:
# Material assignments for each product (based on typical use cases)
MATERIAL_ASSIGNMENTS = {
    "Heat exchanger": "aluminium",      # Aluminium for heat exchangers
    "Fuel tank": "steel",             # Steel for fuel tanks
    "Heterogeneous pipe": "steel",    # Steel pipes
    "Freeze Dryers Shelf": "steel",   # Stainless steel for freeze dryers
    "Battery cooling plate": "aluminium"  # Aluminium for battery cooling
}

In [86]:
# Get only the most used EF 3.1 methods
most_used_indicators = [
    'climate change: total',
    'particulate matter formation',
    'acidification',
    'eutrophication: terrestrial',
    'photochemical oxidant formation: human health',
    'human toxicity: carcinogenic'
]

ef31_methods = [
    m for m in bd.methods
    if ('EF v3.1' in str(m) or 'EF 3.1' in str(m))
    and 'no LT' not in str(m)
    and any(indicator in str(m) for indicator in most_used_indicators)
]

# Physical unit of each impact category, as stored in the method metadata
method_units = {m[2]: bd.Method(m).metadata.get("unit", "") for m in ef31_methods}

print(f"Found {len(ef31_methods)} most-used EF 3.1 methods")
print("Selected EF 3.1 impact categories:")
for m in ef31_methods:
    print(f"  {m}  ->  unit: {method_units[m[2]]}")

Found 8 most-used EF 3.1 methods
Selected EF 3.1 impact categories:
  ('ecoinvent-3.12', 'EF v3.1', 'acidification', 'accumulated exceedance (AE)')  ->  unit: mol H+-Eq
  ('ecoinvent-3.12', 'EF v3.1', 'eutrophication: terrestrial', 'accumulated exceedance (AE)')  ->  unit: mol N-Eq
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, inorganics', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, organics', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'particulate matter formation', 'impact on human health')  ->  unit: disease incidence
  ('ecoinvent-3.12', 'EF v3.1', 'photochemical oxidant formation: human health', 'tropospheric ozone concentration increase')  ->  unit: kg NMVOC-Eq
  ('ecoinvent-3.12', 'EN15804+A2 - Core impact categories 

### Calculation engine for welding process

In [ ]:
def create_welding_exchanges(database_name, welding_params, product_name, elec_key, gas_key):
    """
    Create exchanges from welding parameters.
    Amount is automatically calculated as (minimum + maximum) / 2.
    """
    params = welding_params[product_name]

    # Welding time in seconds (length / speed); the slowest speed gives the longest time
    time_max_s = params["weld_length_m"] / params["welding_speed_m_per_s"]["min"]
    time_min_s = params["weld_length_m"] / params["welding_speed_m_per_s"]["max"]

    # Electricity in kWh: kW * hours
    elec_min = params["power_kW"]["min"] * time_max_s / 3600
    elec_max = params["power_kW"]["max"] * time_min_s / 3600

    elec_amount = (elec_min + elec_max) / 2

    
    # Gas in kg: gas_flow (kg/s) * time (s)
    gas_min = params["gas_flow_kg_per_s"]["min"] * time_max_s
    gas_max = params["gas_flow_kg_per_s"]["max"] * time_min_s
    gas_amount = (gas_min + gas_max) / 2
    
    exchanges = [
        {"input": (database_name, product_name), "amount": 1, "type": "production"},
    ]
    
    # Electricity exchange
    if elec_min != elec_max:
        exchanges.append({
            "input": elec_key,
            "amount": elec_amount,
            "minimum": elec_min,
            "maximum": elec_max,
            "uncertainty_type": 1,
            "type": "technosphere",
        })
    else:
        exchanges.append({
            "input": elec_key,
            "amount": elec_amount,
            "type": "technosphere",
        })
    
    # Gas exchange
    if gas_min != gas_max:
        exchanges.append({
            "input": gas_key,
            "amount": gas_amount,
            "minimum": gas_min,
            "maximum": gas_max,
            "uncertainty_type": 1,
            "type": "technosphere",
        })
    else:
        exchanges.append({

            "input": gas_key,
            "amount": gas_amount,
            "type": "technosphere",
        })
    return exchanges

In [88]:
def calculate_process_cost(process, scenario="amount", cost_factors=None):
    """Calculate total cost for a process based on exchange amounts."""
    if cost_factors is None:
        cost_factors = COST_FACTORS

    total_cost = 0.0

    for exchange in process.exchanges():
        if exchange["type"] == "technosphere":
            # Get amount for chosen scenario
            if scenario == "minimum":
                amount = exchange.get("minimum", exchange.get("min", exchange["amount"]))
            elif scenario == "maximum":
                amount = exchange.get("maximum", exchange.get("max", exchange["amount"]))
            else:
                amount = exchange["amount"]

            # Identify resource type and calculate cost
            input_node = exchange.input
            input_str = str(input_node).lower()

            if "electricity" in input_str:
                total_cost += amount * cost_factors.get('electricity_EUR_per_kWh', 0.29)/1000
            elif "argon" in input_str:
                total_cost += amount * cost_factors.get('argon_EUR_per_kg', 5.87)
            elif "nitrogen" in input_str:
                total_cost += amount * cost_factors.get('nitrogen_EUR_per_kg', 5.35)
    return total_cost

## Use case unit processes with State of the Art technologies

The functional unit is fixed at `1 welded unit` (`1 unit`).

In [89]:
SoA_NAME = "State_of_the_Art"

SoA = bd.Database(SoA_NAME)

In [90]:
# Welding parameters for SoA processes
# Define power (kW), welding speed (m/s), weld length (m), and gas flow (kg/s) ranges
SoA_welding_params = {
    "Heat exchanger": {
        "power_kW": {"min": 1.4, "max": 1.5},
        "welding_speed_m_per_s": {"min": 0.012, "max": 0.014},
        "weld_length_m": 10.0,
        "gas_flow_kg_per_s": {"min": 0.0785, "max": 0.0981},
        "time_m": {"min": 55, "max": 55}
    },
    "Fuel tank": {
        "power_kW": {"min": 1.8, "max": 2.52},
        "welding_speed_m_per_s": {"min": 0.4, "max": 0.5},
        "weld_length_m": 5.0,
        "gas_flow_kg_per_s": {"min": 0.029, "max": 0.036},
        "time_m": {"min": 20, "max": 20}
    },
    "Heterogeneous pipe": {
        "power_kW": {"min": 1.4, "max": 2.1},
        "welding_speed_m_per_s": {"min": 0.8, "max": 1.0},
        "weld_length_m": 20.0,
        "gas_flow_kg_per_s": {"min": 0.002, "max": 0.004},
        "time_m": {"min": 0.5, "max": 0.5}
    },
    "Freeze Dryers Shelf": {
        "power_kW": {"min": 1.8, "max": 2.0},
        "welding_speed_m_per_s": {"min": 0.45, "max": 0.55},
        "weld_length_m": 15.0,
        "gas_flow_kg_per_s": {"min": 0.071, "max": 0.107},
        "time_m": {"min": 10, "max": 10}
    },
    "Battery cooling plate": {
        "power_kW": {"min": 2000, "max": 2000},
        "welding_speed_m_per_s": {"min": 25*70/(3600), "max": 25*100/(3600)},
        "weld_length_m": 25,
        "gas_flow_kg_per_s": {"min": 0.00371, "max": 0.00371},
        "time_m": {"min": 50/60, "max": 1}
    },   }


# Generate SoA_data from welding parameters
SoA_data = {}
for product_name in SoA_welding_params:
    process_type = (
        "controlled atmosphere brazing" if product_name == "Battery cooling plate"
        else "TIG" if product_name == "Fuel tank"
        else "standard laser"
    )
    SoA_data[(SoA_NAME, product_name)] = {
        "name": f"{product_name} joined with {process_type}",
        "unit": "unit",
        "location": "NL",
        "exchanges": create_welding_exchanges(SoA_NAME, SoA_welding_params, product_name, elec.key, gas.key),
    }

In [91]:
SoA.write(SoA_data)

100%|██████████| 5/5 [00:00<00:00, 74631.74it/s]

10:36:40+0100 [info     ] Vacuuming database            


## Use case unit processes with Dynamic Beam Welding

The functional unit is fixed at `1 welded unit` (`1 unit`).

In [92]:
DBW_NAME = "Dynamic_Beam_Welding"
DBW = bd.Database(DBW_NAME)

# Welding parameters with min/max ranges for power and speed
# power_w is electrical input power, not optical laser output power.
dbw_welding_params = {
    "Heat exchanger": {
        "power_kW": {"min": 1.26, "max": 1.35},
        "welding_speed_m_per_s": {"min": 0.0108, "max": 0.0126},
        "weld_length_m": 10.0,
        "gas_flow_kg_per_s": {"min": 0.058875, "max": 0.04905},
        "time_m": {"min": 49.5, "max": 49.5}
    },
    "Fuel tank": {
        "power_kW": {"min": 1.62, "max": 2.268},
        "welding_speed_m_per_s": {"min": 0.36, "max": 0.45},
        "weld_length_m": 5.0,
        "gas_flow_kg_per_s": {"min": 0.02175, "max": 0.018},
        "time_m": {"min": 18.0, "max": 18.0}
    },
    "Heterogeneous pipe": {
        "power_kW": {"min": 1.26, "max": 1.89},
        "welding_speed_m_per_s": {"min": 0.72, "max": 0.9},
        "weld_length_m": 20.0,
        "gas_flow_kg_per_s": {"min": 0.0015, "max": 0.002},
        "time_m": {"min": 0.45, "max": 0.45}
    },
    "Freeze Dryers Shelf": {
        "power_kW": {"min": 1.62, "max": 1.8},
        "welding_speed_m_per_s": {"min": 0.405, "max": 0.495},
        "weld_length_m": 15.0,
        "gas_flow_kg_per_s": {"min": 0.05325, "max": 0.0535},
        "time_m": {"min": 9.0, "max": 9.0}
    },
    "Battery cooling plate": {
        "power_kW": {"min": 8, "max": 10},  # Range around original
        "weld_length_m": 25,
        "welding_speed_m_per_s": {"min": 0.004, "max": 0.004},  # Range around original 0.01
        "gas_flow_kg_per_s": {"min": 0.0, "max": 0.0},
        "time_m": {"min": 50/60, "max": 1}
    },
}

# Generate DBW_data from welding parameters
DBW_data = {}
for product_name in dbw_welding_params:
    process_type = ("Dynamic Beam Welding"
    )
    DBW_data[(DBW_NAME, product_name)] = {
        "name": f"{product_name} joined with {process_type}",
        "unit": "unit",
        "location": "NL",
        "exchanges": create_welding_exchanges(DBW_NAME, dbw_welding_params, product_name, elec.key, gas.key),
    }
    print(DBW_data)
    

{('Dynamic_Beam_Welding', 'Heat exchanger'): {'name': 'Heat exchanger joined with Dynamic Beam Welding', 'unit': 'unit', 'location': 'NL', 'exchanges': [{'input': ('Dynamic_Beam_Welding', 'Heat exchanger'), 'amount': 1, 'type': 'production'}, {'input': ('ecoinvent-3.12-cutoff', '42ec2f4634364ee86921b4c6dfe2de31'), 'amount': 0.3108465608465608, 'minimum': 0.32407407407407407, 'maximum': 0.2976190476190476, 'uncertainty_type': 1, 'type': 'technosphere'}, {'input': ('ecoinvent-3.12-cutoff', '0ff82f527344bc21b1811132b6aef17c'), 'amount': 0.7786871693121693, 'minimum': 0.9085648148148148, 'maximum': 0.6488095238095238, 'uncertainty_type': 1, 'type': 'technosphere'}]}}
{('Dynamic_Beam_Welding', 'Heat exchanger'): {'name': 'Heat exchanger joined with Dynamic Beam Welding', 'unit': 'unit', 'location': 'NL', 'exchanges': [{'input': ('Dynamic_Beam_Welding', 'Heat exchanger'), 'amount': 1, 'type': 'production'}, {'input': ('ecoinvent-3.12-cutoff', '42ec2f4634364ee86921b4c6dfe2de31'), 'amount': 0.

In [93]:
DBW.write(DBW_data)


100%|██████████| 5/5 [00:00<00:00, 84562.58it/s]

10:36:47+0100 [info     ] Vacuuming database            


## LCIA results

In [94]:
def calculate_lcia_scenarios(process_databases, ef31_methods, cost_factors=None):
    if cost_factors is None:
        cost_factors = COST_FACTORS

    scenario_results = []
    category_units = {}

    if not isinstance(process_databases, (list, tuple)):
        process_databases = [process_databases]

    scenarios = ["minimum", "amount", "maximum"]
    scenario_keys = {
        "minimum": ["minimum", "min"],
        "amount": ["amount"],
        "maximum": ["maximum", "max"]
    }

    for database in process_databases:
        for process in database:
            process_exchanges = list(process.exchanges())

            for scenario in scenarios:
                original_amounts = [
                    (exchange, exchange["amount"])
                    for exchange in process_exchanges
                ]

                try:
                    for exchange, original_amount in original_amounts:
                        scenario_amount = original_amount
                        for key in scenario_keys[scenario]:
                            if key in exchange:
                                scenario_amount = exchange[key]
                                break

                        if scenario_amount != original_amount:
                            exchange["amount"] = scenario_amount
                            exchange.save()

                    # Calculate LCIA
                    lca = LCA({process: 1}, method=ef31_methods[0])
                    lca.lci()
                    lca.lcia()

                    # Calculate foreground electricity consumption and total cost
                    electricity_kwh = sum(
                        exchange["amount"]
                        for exchange in process_exchanges
                        if exchange["type"] == "technosphere"
                        and "electricity" in str(exchange.input).lower()
                    )
                    opex = calculate_process_cost(process, scenario, cost_factors)

                    for method in ef31_methods:
                        lca.switch_method(method)
                        lca.lcia()

                        method_tuple = str(method).replace("'", '').strip("()")
                        method_parts = [p.strip() for p in method_tuple.split(",")]
                        impact_category = method_parts[2] if len(method_parts) > 2 else str(method)

                        category_units[impact_category] = bd.Method(method).metadata.get("unit", "")

                        scenario_results.append({
                            "Database": database.filename,
                            "Process": process["name"],
                            "Key": process.key[1],
                            "Scenario": scenario,
                            "Method": str(method),
                            "Impact Category": impact_category,
                            "Amount": lca.score,
                            "Electricity Consumption (kWh)": electricity_kwh,
                            "Consumable costs (EUR)": opex
                        })
                finally:
                    for exchange, original_amount in original_amounts:
                        if exchange["amount"] != original_amount:
                            exchange["amount"] = original_amount
                            exchange.save()

    # Create DataFrame and add costs
    scenario_df = pd.DataFrame(scenario_results)

    pivot_scenario_df = scenario_df.pivot_table(
        index=["Database", "Process", "Key", "Scenario"],
        columns="Impact Category",
        values="Amount",
        aggfunc="sum"
    ).reset_index()

    result_index = ["Database", "Process", "Key", "Scenario"]
    resource_df = scenario_df[result_index + ["Electricity Consumption (kWh)", "Consumable costs (EUR)"]].drop_duplicates(subset=result_index)

    final_df = pd.merge(pivot_scenario_df, resource_df,
                       on=result_index, how="left")

    # Column headers with units for display and export; final_df keeps plain names
    export_df = final_df.rename(columns={
        cat: f"{cat} [{unit}]" for cat, unit in category_units.items() if unit
    })

    print("\n\n=== EF 3.1 LCIA Results with Foreground Electricity Consumption and Costs ===")
    print(export_df.to_markdown(index=False, floatfmt=".4e"))

    # Save results as CSV and JSON
    if len(process_databases) == 1:
        output_file = f"{process_databases[0].filename.split('.')[0]}_LCIA_results.csv"
    else:
        db_names = "_".join([db.filename.split('.')[0] for db in process_databases])
        output_file = f"results/{db_names}_LCIA_results.csv"

    json_file = output_file.rsplit(".", 1)[0] + ".json"

    export_df.to_csv(output_file, index=False)
    export_df.to_json(json_file, orient="records", indent=2)

    print(f"\nScenario results saved to {output_file} and {json_file}")

    return final_df

In [95]:
# calculate_lcia_scenarios([SoA, DBW], ef31_methods)
# %run results/generate_lca_graphs.py

In [96]:
# Capture LCIA results (re-uses calculate_lcia_scenarios output)
lcia_df = calculate_lcia_scenarios([SoA, DBW], ef31_methods)

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 7.65e+12)
  warnings.warn(msg, UmfpackWarning)




=== EF 3.1 LCIA Results with Foreground Electricity Consumption and Costs ===
| Database                      | Process                                                         | Key                   | Scenario   |   acidification [mol H+-Eq] |   climate change: total (EF v3.1 - IPCC 2021) [kg CO2-Eq] |   eutrophication: terrestrial [mol N-Eq] |   human toxicity: carcinogenic [CTUh] |   particulate matter formation [disease incidence] |   photochemical oxidant formation: human health [kg NMVOC-Eq] |   Electricity Consumption (kWh) |   Consumable costs (EUR) |
|:------------------------------|:----------------------------------------------------------------|:----------------------|:-----------|----------------------------:|----------------------------------------------------------:|-----------------------------------------:|--------------------------------------:|---------------------------------------------------:|--------------------------------------------------------------:|------

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.patches import Patch

plt.close("all")

use_cases = list(SoA_welding_params.keys())
result_index = ["Database", "Process", "Key", "Scenario"]
indicator_cols = [
    c for c in lcia_df.columns
    if c not in result_index
]
print(indicator_cols)

all_technologies = set()
for db_name in [SoA_NAME, DBW_NAME]:
    db_data = lcia_df[lcia_df["Database"].str.startswith(db_name)]
    for process_name in db_data["Process"].unique():
        all_technologies.add(process_name.split(" joined with ", 1)[-1])

technologies = sorted(all_technologies)
palette = plt.get_cmap("tab10")
tech_color_map = {
    tech: palette(i % 10)
    for i, tech in enumerate(technologies)
}

def plot_indicator(indicator):
    fig, axes = plt.subplots(3, 3, figsize=(16, 14), squeeze=False)
    axes = axes.flatten()

    unit = method_units.get(indicator, "unit")

    x = np.arange(2)

    for ax, use_case in zip(axes, use_cases):
        case_rows = lcia_df[lcia_df["Key"] == use_case]
        tick_labels = []

        for db_name in [SoA_NAME, DBW_NAME]:
            db_rows = case_rows[case_rows["Database"].str.startswith(db_name)]

            if db_rows.empty:
                tick_labels.append("")
                continue

            tech = db_rows.iloc[0]["Process"].split(" joined with ", 1)[-1]
            tick_labels.append(tech)

            amount_rows = db_rows[db_rows["Scenario"] == "amount"]
            min_rows = db_rows[db_rows["Scenario"] == "minimum"]
            max_rows = db_rows[db_rows["Scenario"] == "maximum"]

            if amount_rows.empty or min_rows.empty or max_rows.empty:
                continue

            amount = float(amount_rows[indicator].iloc[0])
            low = min(float(min_rows[indicator].iloc[0]),
                      float(max_rows[indicator].iloc[0]), amount)
            high = max(float(min_rows[indicator].iloc[0]),
                       float(max_rows[indicator].iloc[0]), amount)

            position = 0 if db_name == SoA_NAME else 1
            ax.bar(
                position,
                amount,
                width=0.55,
                color=tech_color_map[tech],
                yerr=[[amount - low], [high - amount]],
                capsize=5,
            )

        ax.set_title(use_case, fontsize=11, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, fontsize=9)
        ax.set_ylabel(f"{indicator} - {unit}", fontsize=9)
        ax.grid(axis="y", alpha=0.3, linestyle="--")

    for ax in axes[len(use_cases):]:
        ax.axis("off")

    handles = [
        Patch(facecolor=tech_color_map[tech], label=tech)
        for tech in technologies
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.95),
        ncol=len(handles),
        fontsize=10,
        frameon=False,
    )
    fig.suptitle(indicator, fontsize=16, fontweight="bold", y=0.97)
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.93))
    return fig

indicator_dropdown = widgets.Dropdown(
    options=indicator_cols,
    description="Indicator:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "initial"},
)
output = widgets.Output()

def on_change(change):
    if change["name"] == "value":
        with output:
            clear_output(wait=True)
            fig = plot_indicator(change["new"])
            display(fig)
            plt.close(fig)

indicator_dropdown.observe(on_change, names="value")
display(widgets.VBox([indicator_dropdown, output]))
on_change({"name": "value", "new": indicator_dropdown.value})


['acidification', 'climate change: total (EF v3.1 - IPCC 2021)', 'eutrophication: terrestrial', 'human toxicity: carcinogenic', 'particulate matter formation', 'photochemical oxidant formation: human health', 'Electricity Consumption (kWh)', 'Consumable costs (EUR)']


## Regional analysis: EU electricity markets
Replaces the RER electricity market with each member state's own ecoinvent market and maps the resulting impacts per welded unit.

In [ ]:
# Impact per kWh of each EU member state's own electricity market (and of the argon input)
EU27 = {"AT":"AUT","BE":"BEL","BG":"BGR","HR":"HRV","CY":"CYP","CZ":"CZE","DK":"DNK","EE":"EST","FI":"FIN",
        "FR":"FRA","DE":"DEU","GR":"GRC","HU":"HUN","IE":"IRL","IT":"ITA","LV":"LVA","LT":"LTU","LU":"LUX",
        "MT":"MLT","NL":"NLD","PL":"POL","PT":"PRT","RO":"ROU","SK":"SVK","SI":"SVN","ES":"ESP","SE":"SWE"}

def unit_impacts(act):
    lca = LCA({act: 1}, method=ef31_methods[0]); lca.lci()
    out = {}
    for m in ef31_methods:
        lca.switch_method(m); lca.lcia(); out[m[2]] = lca.score
    return out

grid = {"RER": unit_impacts(elec)}   # baseline used above
for iso2 in EU27:
    market = next(a for a in ei if a["name"] == "market for electricity, medium voltage" and a["location"] == iso2)
    grid[iso2] = unit_impacts(market)
gas_imp = unit_impacts(gas)

# Foreground processes only have electricity + gas inputs, so the score is linear in both
rows = []
for db in (SoA, DBW):
    for proc in db:
        tech = proc["name"].split(" joined with ")[-1]
        kwh = sum(e["amount"] for e in proc.technosphere() if "electricity" in e.input["name"])
        kg  = sum(e["amount"] for e in proc.technosphere() if "electricity" not in e.input["name"])
        for c, imp in grid.items():
            for ind in imp:
                rows.append({"use_case": proc.key[1], "technology": tech, "country": c, "indicator": ind,
                             "impact": kwh * imp[ind] + kg * gas_imp[ind]})
regional_df = pd.DataFrame(rows)
regional_df.head()


In [ ]:
import plotly.express as px

uc_opts   = list(SoA_welding_params)
ind_opts  = [m[2] for m in ef31_methods]
w_ind  = widgets.Dropdown(options=ind_opts, description="Indicator")
w_uc   = widgets.Dropdown(options=uc_opts, description="Use case")
w_tech = widgets.Dropdown(options=["State of the Art", "Dynamic Beam Welding"], description="Technology")
w_view = widgets.Dropdown(options=["Absolute", "Change vs RER (%)"], description="View")
out = widgets.Output()

def draw(_=None):
    d = regional_df[(regional_df.use_case == w_uc.value) & (regional_df.indicator == w_ind.value)]
    is_dbw = d.technology == "Dynamic Beam Welding"
    d = d[is_dbw if w_tech.value == "Dynamic Beam Welding" else ~is_dbw].set_index("country")["impact"]
    rer = d["RER"]; d = d.drop("RER")
    df = pd.DataFrame({"iso3": [EU27[c] for c in d.index], "country": d.index, "value": d.values})
    unit = method_units.get(w_ind.value, "")
    if w_view.value == "Absolute":
        df["plot"] = df["value"]; kw = dict(color_continuous_scale="YlOrRd"); label = unit
    else:
        df["plot"] = (df["value"] / rer - 1) * 100
        m = df["plot"].abs().max(); kw = dict(color_continuous_scale="RdBu_r", range_color=(-m, m)); label = "% vs RER"
    fig = px.choropleth(df, locations="iso3", color="plot", hover_name="country", hover_data={"iso3": False, "value": ":.3g"},
                        scope="europe", labels={"plot": label}, **kw)
    fig.update_geos(lonaxis_range=[-25, 35], lataxis_range=[34, 71])
    fig.update_layout(height=600, margin=dict(l=0, r=0, t=40, b=0),
                      title=f"{w_uc.value} / {w_tech.value}: {w_ind.value} per welded unit (RER baseline {rer:.3g} {unit})")
    with out:
        out.clear_output(wait=True); fig.show()

for w in (w_ind, w_uc, w_tech, w_view): w.observe(draw, names="value")
display(widgets.VBox([widgets.HBox([w_ind, w_uc]), widgets.HBox([w_tech, w_view]), out]))
draw()
